In [0]:
%pip install kaggle scikit-learn optuna category_encoders imbalanced-learn matplotlib seaborn xgboost lightgbm

In [0]:
dbutils.library.restartPython()

In [0]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pyspark.sql import functions as F
from pyspark.sql.types import *
import mlflow
import mlflow.sklearn
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import roc_auc_score, classification_report, confusion_matrix, roc_curve
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
import category_encoders as ce
import optuna
from datetime import datetime

# Configuración de visualización
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

print("✓ Librerías importadas exitosamente")

In [0]:
# Configuración de credenciales de Kaggle
# NOTA: Necesitas configurar tus credenciales de Kaggle
# Opción 1: Subir kaggle.json a /Workspace/Users/caroflorez000@gmail.com/.kaggle/
# Opción 2: Configurar variables de entorno KAGGLE_USERNAME y KAGGLE_KEY

import zipfile
import subprocess

# Crear directorio para datos
data_path = '/tmp/credit_risk_data/'
os.makedirs(data_path, exist_ok=True)

# Descargar dataset de Kaggle
try:
    # Intentar descargar usando kaggle API
    subprocess.run([
        'kaggle', 'datasets', 'download', 
        '-d', 'laotse/credit-risk-dataset',
        '-p', data_path,
        '--unzip'
    ], check=True)
    print("✓ Dataset descargado exitosamente")
except Exception as e:
    print(f"⚠ Error al descargar: {e}")
    print("\nPara configurar Kaggle API:")
    print("1. Ve a https://www.kaggle.com/settings/account")
    print("2. Crea un nuevo API token (descargará kaggle.json)")
    print("3. Sube el archivo o configura las credenciales manualmente")
    print("\nAlternativamente, descarga manualmente el archivo y súbelo a:", data_path)

In [0]:
# BRONZE LAYER: Datos crudos sin transformaciones
print("=" * 60)
print("BRONZE LAYER - Datos Crudos")
print("=" * 60)

# Leer archivo CSV
csv_files = [f for f in os.listdir(data_path) if f.endswith('.csv')]
if csv_files:
    csv_file = os.path.join(data_path, csv_files[0])
    # En serverless, spark.read no puede acceder a /tmp/
    # Usar pandas para leer, luego convertir a Spark
    df_bronze_pd = pd.read_csv(csv_file)
    df_bronze = spark.createDataFrame(df_bronze_pd)
    
    # Guardar como tabla Delta en Unity Catalog (serverless no permite /FileStore/)
    spark.sql("CREATE SCHEMA IF NOT EXISTS credit_risk")
    df_bronze.write.format('delta').mode('overwrite').saveAsTable('credit_risk.bronze')
    
    print(f"✓ Datos cargados: {df_bronze.count()} registros")
    print(f"✓ Columnas: {len(df_bronze.columns)}")
    print("\nEsquema:")
    df_bronze.printSchema()
    
    print("\nPrimeras filas:")
    display(df_bronze.limit(5))
else:
    print("⚠ No se encontró archivo CSV. Por favor descarga manualmente desde:")
    print("https://www.kaggle.com/datasets/laotse/credit-risk-dataset/data")

In [0]:
# SILVER LAYER: Limpieza, validación y estandarización
print("=" * 60)
print("SILVER LAYER - Limpieza y Validación")
print("=" * 60)

# Leer desde Bronze tabla registrada
df_silver = spark.table('credit_risk.bronze')

# Análisis de calidad de datos
print("\n📊 Análisis de Calidad de Datos:")
print("-" * 60)

# Contar valores nulos
print("\nValores nulos por columna:")
null_counts = df_silver.select([F.sum(F.col(c).isNull().cast("int")).alias(c) for c in df_silver.columns])
display(null_counts)

# Eliminar duplicados
initial_count = df_silver.count()
df_silver = df_silver.dropDuplicates()
duplicates_removed = initial_count - df_silver.count()
print(f"\n✓ Duplicados eliminados: {duplicates_removed}")

# Renombrar columnas para consistencia (snake_case)
for col in df_silver.columns:
    new_col = col.lower().replace(' ', '_').replace('-', '_')
    if new_col != col:
        df_silver = df_silver.withColumnRenamed(col, new_col)

print(f"\n✓ Columnas estandarizadas")

# Guardar Silver layer como tabla Delta (serverless no permite /FileStore/)
df_silver.write.format('delta').mode('overwrite').saveAsTable('credit_risk.silver')

print(f"\n✓ Silver layer guardada: {df_silver.count()} registros")
print("\nVista previa:")
display(df_silver.limit(5))

In [0]:
# EXPLORATORY DATA ANALYSIS
print("=" * 60)
print("ANÁLISIS EXPLORATORIO DE DATOS")
print("=" * 60)

# Convertir a Pandas para EDA
df_pd = df_silver.toPandas()

print(f"\n📋 Dimensiones: {df_pd.shape[0]} filas x {df_pd.shape[1]} columnas")
print("\n📄 Información del Dataset:")
print(df_pd.info())

print("\n📊 Estadísticas Descriptivas:")
display(df_pd.describe())

# Identificar variable target (loan_status es la variable de default/riesgo)
if 'loan_status' in df_pd.columns:
    target_col = 'loan_status'
    print(f"\n🎯 Variable Target identificada: {target_col}")
    print("\nDistribución de la variable target:")
    print(df_pd[target_col].value_counts())
    print("\nProporción:")
    print(df_pd[target_col].value_counts(normalize=True))
else:
    print("⚠ Variable target no identificada claramente")
    print("Columnas disponibles:", df_pd.columns.tolist())

In [0]:
# Tabla de tipos de variables
print("\n📋 TABLA DE TIPOS DE VARIABLES")
print("=" * 60)

variable_types = []

for col in df_pd.columns:
    dtype = df_pd[col].dtype
    unique_count = df_pd[col].nunique()
    
    # Clasificar tipo de variable
    if col == target_col:
        var_type = 'Target (Binaria)'
    elif dtype in ['int64', 'float64']:
        if unique_count < 10:
            var_type = 'Categórica Ordinal (numérica)'
        else:
            var_type = 'Numérica Continua'
    elif dtype == 'object':
        if unique_count < 20:
            var_type = 'Categórica Nominal'
        else:
            var_type = 'Texto/Alta Cardinalidad'
    else:
        var_type = str(dtype)
    
    variable_types.append({
        'Variable': col,
        'Tipo de Dato': str(dtype),
        'Valores Únicos': unique_count,
        'Tipo de Variable': var_type,
        '% Nulos': f"{(df_pd[col].isna().sum() / len(df_pd) * 100):.2f}%"
    })

variable_types_df = pd.DataFrame(variable_types)
display(variable_types_df)

In [0]:
# Visualización de la distribución del target
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Gráfico de barras
df_pd[target_col].value_counts().plot(kind='bar', ax=axes[0], color=['#2ecc71', '#e74c3c'])
axes[0].set_title('Distribución de Riesgo de Crédito', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Estado del Préstamo')
axes[0].set_ylabel('Frecuencia')
axes[0].tick_params(axis='x', rotation=45)

# Gráfico de pastel
df_pd[target_col].value_counts().plot(kind='pie', ax=axes[1], autopct='%1.1f%%', 
                                       colors=['#2ecc71', '#e74c3c'], startangle=90)
axes[1].set_ylabel('')
axes[1].set_title('Proporción de Riesgo', fontsize=14, fontweight='bold')

plt.tight_layout()
plt.show()

# Verificar desbalanceo
class_counts = df_pd[target_col].value_counts()
imbalance_ratio = class_counts.max() / class_counts.min()
print(f"\n⚠ Ratio de desbalanceo: {imbalance_ratio:.2f}:1")
if imbalance_ratio > 3:
    print("→ Dataset desbalanceado. Considerar técnicas de balanceo.")

In [0]:
# Matriz de correlación para variables numéricas
print("\n🔗 MATRIZ DE CORRELACIÓN")
print("=" * 60)

# Seleccionar solo columnas numéricas
numeric_cols = df_pd.select_dtypes(include=[np.number]).columns.tolist()

# Codificar target si es necesario para correlación
df_corr = df_pd[numeric_cols].copy()

if target_col in df_pd.columns and df_pd[target_col].dtype == 'object':
    # Codificar target
    from sklearn.preprocessing import LabelEncoder
    le = LabelEncoder()
    df_corr[target_col] = le.fit_transform(df_pd[target_col])

# Calcular correlación
corr_matrix = df_corr.corr()

# Visualizar correlograma
plt.figure(figsize=(12, 10))
sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='coolwarm', center=0,
            square=True, linewidths=1, cbar_kws={"shrink": 0.8})
plt.title('Correlograma - Relaciones entre Variables', fontsize=16, fontweight='bold', pad=20)
plt.tight_layout()
plt.show()

# Correlaciones con el target
if target_col in df_corr.columns:
    print("\n🎯 Correlaciones con la variable Target:")
    target_corr = corr_matrix[target_col].sort_values(ascending=False)
    print(target_corr)
    
    # Visualizar top correlaciones
    top_n = min(10, len(target_corr) - 1)
    target_corr_top = target_corr[target_corr.index != target_col].head(top_n)
    
    plt.figure(figsize=(10, 6))
    target_corr_top.plot(kind='barh', color=['#e74c3c' if x < 0 else '#3498db' for x in target_corr_top])
    plt.title(f'Top {top_n} Variables Correlacionadas con {target_col}', fontsize=14, fontweight='bold')
    plt.xlabel('Coeficiente de Correlación')
    plt.ylabel('Variable')
    plt.axvline(x=0, color='black', linestyle='--', linewidth=0.8)
    plt.tight_layout()
    plt.show()

In [0]:
# Visualizaciones de relación entre features numéricas y target
print("\n📉 RELACIÓN DE FEATURES CON TARGET")
print("=" * 60)

# Seleccionar las 4 variables numéricas más correlacionadas
if target_col in df_corr.columns:
    top_features = target_corr[target_corr.index != target_col].abs().sort_values(ascending=False).head(4).index.tolist()
    
    fig, axes = plt.subplots(2, 2, figsize=(15, 12))
    axes = axes.ravel()
    
    for idx, feature in enumerate(top_features):
        if feature in df_pd.columns:
            # Box plot para cada feature por categoría de target
            df_pd.boxplot(column=feature, by=target_col, ax=axes[idx])
            axes[idx].set_title(f'{feature} vs {target_col}', fontsize=12, fontweight='bold')
            axes[idx].set_xlabel(target_col)
            axes[idx].set_ylabel(feature)
            plt.sca(axes[idx])
            plt.xticks(rotation=45)
    
    plt.tight_layout()
    plt.show()
    
    # Visualización adicional: Distribución de variables categóricas
    categorical_cols = df_pd.select_dtypes(include=['object']).columns.tolist()
    if target_col in categorical_cols:
        categorical_cols.remove(target_col)
    
    if len(categorical_cols) > 0:
        print("\n📊 Distribución de Variables Categóricas por Target:")
        n_cat = min(3, len(categorical_cols))
        
        fig, axes = plt.subplots(1, n_cat, figsize=(15, 5))
        if n_cat == 1:
            axes = [axes]
        
        for idx, cat_col in enumerate(categorical_cols[:n_cat]):
            pd.crosstab(df_pd[cat_col], df_pd[target_col], normalize='index').plot(
                kind='bar', stacked=False, ax=axes[idx], color=['#2ecc71', '#e74c3c']
            )
            axes[idx].set_title(f'{cat_col} vs {target_col}', fontsize=12, fontweight='bold')
            axes[idx].set_xlabel(cat_col)
            axes[idx].set_ylabel('Proporción')
            axes[idx].legend(title=target_col)
            axes[idx].tick_params(axis='x', rotation=45)
        
        plt.tight_layout()
        plt.show()

In [0]:
# GOLD LAYER: Features listas para Machine Learning
print("=" * 60)
print("GOLD LAYER - Features para Machine Learning")
print("=" * 60)

# Preparar datos para ML
df_gold = df_pd.copy()

# Identificar columnas por tipo
numeric_features = df_gold.select_dtypes(include=[np.number]).columns.tolist()
categorical_features = df_gold.select_dtypes(include=['object']).columns.tolist()

# Remover target de features
if target_col in numeric_features:
    numeric_features.remove(target_col)
if target_col in categorical_features:
    categorical_features.remove(target_col)

print(f"\n🔢 Features numéricas: {len(numeric_features)}")
print(numeric_features)
print(f"\n🔤 Features categóricas: {len(categorical_features)}")
print(categorical_features)

# Preparar X y y
X = df_gold.drop(columns=[target_col])
y = df_gold[target_col]

# Codificar variable target si es necesario
from sklearn.preprocessing import LabelEncoder
if y.dtype == 'object':
    le_target = LabelEncoder()
    y = le_target.fit_transform(y)
    print(f"\n✓ Target codificado: {le_target.classes_}")
    print(f"  0 = {le_target.classes_[0]}")
    print(f"  1 = {le_target.classes_[1]}")

print(f"\n🎯 Shape de X: {X.shape}")
print(f"🎯 Shape de y: {y.shape}")
print(f"\n✓ Gold layer preparada para modelado")

In [0]:
# Split de datos: 70% train, 15% validation, 15% test
print("\n📦 DIVISIÓN DE DATOS")
print("=" * 60)

# Primera división: 70% train, 30% temp
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)

# Segunda división: 15% validation, 15% test
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.5, random_state=42, stratify=y_temp
)

print(f"\nConjunto de entrenamiento: {X_train.shape[0]} muestras ({X_train.shape[0]/len(X)*100:.1f}%)")
print(f"Conjunto de validación: {X_val.shape[0]} muestras ({X_val.shape[0]/len(X)*100:.1f}%)")
print(f"Conjunto de prueba: {X_test.shape[0]} muestras ({X_test.shape[0]/len(X)*100:.1f}%)")

print("\n✓ Justificación del split:")
print("  - 70% training: Suficiente datos para entrenar modelos complejos")
print("  - 15% validation: Para tuning de hiperparámetros")
print("  - 15% test: Para evaluación final imparcial")
print("  - Estratificado: Mantiene proporción de clases en cada conjunto")

In [0]:
# Crear pipeline de preprocessing
print("\n🛠 PIPELINE DE PREPROCESAMIENTO")
print("=" * 60)

# Imputación y escalado para numéricas
numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='mean')),
    ('scaler', StandardScaler())
])

# One-hot encoding para categóricas
categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='constant', fill_value='missing')),
    ('onehot', ce.OneHotEncoder(handle_unknown='indicator', use_cat_names=True))
])

# Combinar transformadores
preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_features),
        ('cat', categorical_transformer, categorical_features)
    ])

print("✓ Pipeline de preprocessing creado:")
print("  - Numéricas: Imputación (media) + Escalado estándar")
print("  - Categóricas: Imputación (constante) + One-hot encoding")

# Aplicar transformación
X_train_processed = preprocessor.fit_transform(X_train)
X_val_processed = preprocessor.transform(X_val)
X_test_processed = preprocessor.transform(X_test)

print(f"\n✓ Datos transformados:")
print(f"  Train shape: {X_train_processed.shape}")
print(f"  Val shape: {X_val_processed.shape}")
print(f"  Test shape: {X_test_processed.shape}")

In [0]:
# SELECCIÓN DE VARIABLES
print("=" * 60)
print("SELECCIÓN DE VARIABLES")
print("=" * 60)

from sklearn.feature_selection import SelectKBest, f_classif, mutual_info_classif
from sklearn.ensemble import RandomForestClassifier

# Método 1: ANOVA F-test
print("\n1️⃣ Método 1: ANOVA F-test (SelectKBest)")
print("-" * 60)
selector_anova = SelectKBest(score_func=f_classif, k='all')
selector_anova.fit(X_train_processed, y_train)

# Obtener scores
feature_names = (numeric_features + 
                [f"{col}_{val}" for col in categorical_features 
                 for val in X_train[col].unique()][:X_train_processed.shape[1]-len(numeric_features)])

if len(feature_names) != X_train_processed.shape[1]:
    feature_names = [f"feature_{i}" for i in range(X_train_processed.shape[1])]

anova_scores = pd.DataFrame({
    'Feature': feature_names,
    'ANOVA_F_Score': selector_anova.scores_
}).sort_values('ANOVA_F_Score', ascending=False)

print("\nTop 10 variables por ANOVA F-score:")
print(anova_scores.head(10))

# Método 2: Mutual Information
print("\n2️⃣ Método 2: Mutual Information")
print("-" * 60)
mi_scores = mutual_info_classif(X_train_processed, y_train, random_state=42)
mi_df = pd.DataFrame({
    'Feature': feature_names,
    'MI_Score': mi_scores
}).sort_values('MI_Score', ascending=False)

print("\nTop 10 variables por Mutual Information:")
print(mi_df.head(10))

# Método 3: Random Forest Feature Importance
print("\n3️⃣ Método 3: Random Forest Feature Importance")
print("-" * 60)
rf_selector = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
rf_selector.fit(X_train_processed, y_train)

rf_importance = pd.DataFrame({
    'Feature': feature_names,
    'RF_Importance': rf_selector.feature_importances_
}).sort_values('RF_Importance', ascending=False)

print("\nTop 10 variables por Random Forest Importance:")
print(rf_importance.head(10))

# Combinar métodos
print("\n🏆 RANKING COMBINADO DE VARIABLES")
print("=" * 60)

# Normalizar scores a escala 0-1
anova_scores['ANOVA_Norm'] = (anova_scores['ANOVA_F_Score'] - anova_scores['ANOVA_F_Score'].min()) / (anova_scores['ANOVA_F_Score'].max() - anova_scores['ANOVA_F_Score'].min())
mi_df['MI_Norm'] = (mi_df['MI_Score'] - mi_df['MI_Score'].min()) / (mi_df['MI_Score'].max() - mi_df['MI_Score'].min())
rf_importance['RF_Norm'] = (rf_importance['RF_Importance'] - rf_importance['RF_Importance'].min()) / (rf_importance['RF_Importance'].max() - rf_importance['RF_Importance'].min())

# Merge
combined_scores = anova_scores[['Feature', 'ANOVA_Norm']].merge(
    mi_df[['Feature', 'MI_Norm']], on='Feature'
).merge(
    rf_importance[['Feature', 'RF_Norm']], on='Feature'
)

# Calcular score promedio
combined_scores['Combined_Score'] = (combined_scores['ANOVA_Norm'] + 
                                      combined_scores['MI_Norm'] + 
                                      combined_scores['RF_Norm']) / 3

combined_scores = combined_scores.sort_values('Combined_Score', ascending=False)

print("\nTop 15 variables por score combinado:")
display(combined_scores.head(15))

# Visualizar top variables
plt.figure(figsize=(12, 8))
top_features_viz = combined_scores.head(15)
plt.barh(range(len(top_features_viz)), top_features_viz['Combined_Score'], color='#3498db')
plt.yticks(range(len(top_features_viz)), top_features_viz['Feature'])
plt.xlabel('Score Combinado (normalizado)', fontsize=12)
plt.title('Top 15 Variables Seleccionadas', fontsize=14, fontweight='bold')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

# Seleccionar top K features
top_k = 15
top_features = combined_scores.head(top_k)['Feature'].tolist()
print(f"\n✓ Seleccionadas {top_k} features para modelado")

In [0]:
# ENTRENAMIENTO DE MÚltIPLES MODELOS
print("=" * 60)
print("ENTRENAMIENTO Y COMPARACIÓN DE MODELOS")
print("=" * 60)

# Configurar MLflow
mlflow.set_experiment("/Users/caroflorez000@gmail.com/credit_risk_scoring")

# Definir modelos a comparar
models = {
    'Logistic Regression': LogisticRegression(random_state=42, max_iter=1000),
    'Random Forest': RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1),
    'Gradient Boosting': GradientBoostingClassifier(n_estimators=100, random_state=42),
    'XGBoost': XGBClassifier(n_estimators=100, random_state=42, eval_metric='logloss', use_label_encoder=False),
    'LightGBM': LGBMClassifier(n_estimators=100, random_state=42, verbose=-1)
}

# Almacenar resultados
results = []

print("\n🚀 Entrenando modelos...\n")

for model_name, model in models.items():
    print(f"\n{'='*60}")
    print(f"Entrenando: {model_name}")
    print(f"{'='*60}")
    
    with mlflow.start_run(run_name=model_name):
        # Registrar parámetros
        mlflow.log_params(model.get_params())
        
        # Entrenar modelo
        start_time = datetime.now()
        model.fit(X_train_processed, y_train)
        training_time = (datetime.now() - start_time).total_seconds()
        
        # Predicciones
        y_train_pred = model.predict(X_train_processed)
        y_train_proba = model.predict_proba(X_train_processed)[:, 1]
        
        y_val_pred = model.predict(X_val_processed)
        y_val_proba = model.predict_proba(X_val_processed)[:, 1]
        
        # Métricas
        train_auc = roc_auc_score(y_train, y_train_proba)
        val_auc = roc_auc_score(y_val, y_val_proba)
        
        # Registrar métricas
        mlflow.log_metric("train_auc", train_auc)
        mlflow.log_metric("val_auc", val_auc)
        mlflow.log_metric("training_time_seconds", training_time)
        
        # Registrar modelo
        mlflow.sklearn.log_model(model, "model")
        
        # Almacenar resultados
        results.append({
            'Model': model_name,
            'Train AUC': train_auc,
            'Validation AUC': val_auc,
            'Overfitting': train_auc - val_auc,
            'Training Time (s)': training_time
        })
        
        print(f"✓ Train AUC: {train_auc:.4f}")
        print(f"✓ Validation AUC: {val_auc:.4f}")
        print(f"✓ Tiempo de entrenamiento: {training_time:.2f}s")

print(f"\n\n{'='*60}")
print("✅ Todos los modelos entrenados exitosamente")
print(f"{'='*60}")

In [0]:
# COMPARACIÓN DE MODELOS
print("=" * 60)
print("🏆 COMPARACIÓN DE RENDIMIENTO DE MODELOS")
print("=" * 60)

# Crear DataFrame de resultados
results_df = pd.DataFrame(results).sort_values('Validation AUC', ascending=False)

print("\n📋 Tabla de Resultados:")
display(results_df.style.background_gradient(subset=['Validation AUC'], cmap='Greens'))

# Identificar mejor modelo
best_model_name = results_df.iloc[0]['Model']
best_val_auc = results_df.iloc[0]['Validation AUC']

print(f"\n🌟 MEJOR MODELO: {best_model_name}")
print(f"🎯 Validation AUC: {best_val_auc:.4f}")

# Visualización 1: Comparación de AUC
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Gráfico de barras comparativo
ax1 = axes[0]
x_pos = np.arange(len(results_df))
width = 0.35

ax1.bar(x_pos - width/2, results_df['Train AUC'], width, label='Train AUC', color='#3498db', alpha=0.8)
ax1.bar(x_pos + width/2, results_df['Validation AUC'], width, label='Validation AUC', color='#2ecc71', alpha=0.8)

ax1.set_xlabel('Modelo', fontsize=12)
ax1.set_ylabel('AUC Score', fontsize=12)
ax1.set_title('Comparación de AUC: Train vs Validation', fontsize=14, fontweight='bold')
ax1.set_xticks(x_pos)
ax1.set_xticklabels(results_df['Model'], rotation=45, ha='right')
ax1.legend()
ax1.set_ylim([0.5, 1.0])
ax1.grid(axis='y', alpha=0.3)

# Gráfico de ranking
ax2 = axes[1]
results_sorted = results_df.sort_values('Validation AUC', ascending=True)
colors = ['#f39c12' if model == best_model_name else '#95a5a6' for model in results_sorted['Model']]
ax2.barh(results_sorted['Model'], results_sorted['Validation AUC'], color=colors, alpha=0.8)
ax2.set_xlabel('Validation AUC', fontsize=12)
ax2.set_title('Ranking de Modelos por Validation AUC', fontsize=14, fontweight='bold')
ax2.set_xlim([0.5, 1.0])
ax2.grid(axis='x', alpha=0.3)

# Destacar mejor modelo
for i, (idx, row) in enumerate(results_sorted.iterrows()):
    if row['Model'] == best_model_name:
        ax2.text(row['Validation AUC'] + 0.01, i, f" ★ {row['Validation AUC']:.4f}", 
                va='center', fontweight='bold', color='#f39c12', fontsize=11)

plt.tight_layout()
plt.show()

# Visualización 2: Overfitting
print("\n📉 Análisis de Overfitting:")
plt.figure(figsize=(10, 6))
results_sorted_overfit = results_df.sort_values('Overfitting', ascending=True)
colors_overfit = ['#e74c3c' if x > 0.05 else '#2ecc71' for x in results_sorted_overfit['Overfitting']]
plt.barh(results_sorted_overfit['Model'], results_sorted_overfit['Overfitting'], color=colors_overfit, alpha=0.8)
plt.xlabel('Overfitting (Train AUC - Val AUC)', fontsize=12)
plt.title('Análisis de Overfitting por Modelo', fontsize=14, fontweight='bold')
plt.axvline(x=0.05, color='orange', linestyle='--', linewidth=2, label='Umbral aceptable (0.05)')
plt.legend()
plt.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.show()

In [0]:
# OPTIMIZACIÓN DE HIPERPARÁMETROS DEL MEJOR MODELO
print("=" * 60)
print(f"⚙️ OPTIMIZACIÓN DE HIPERPARÁMETROS: {best_model_name}")
print("=" * 60)

# Definir función objetivo para Optuna
def objective(trial):
    # Definir espacio de búsqueda según el mejor modelo
    if best_model_name == 'Random Forest':
        params = {
            'n_estimators': trial.suggest_int('n_estimators', 100, 300),
            'max_depth': trial.suggest_int('max_depth', 5, 30),
            'min_samples_split': trial.suggest_int('min_samples_split', 2, 20),
            'min_samples_leaf': trial.suggest_int('min_samples_leaf', 1, 10),
            'max_features': trial.suggest_categorical('max_features', ['sqrt', 'log2']),
            'random_state': 42,
            'n_jobs': -1
        }
        model = RandomForestClassifier(**params)
    
    elif best_model_name == 'XGBoost':
        params = {
            'n_estimators': trial.suggest_int('n_estimators', 100, 300),
            'max_depth': trial.suggest_int('max_depth', 3, 10),
            'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3),
            'subsample': trial.suggest_float('subsample', 0.6, 1.0),
            'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 1.0),
            'random_state': 42,
            'eval_metric': 'logloss',
            'use_label_encoder': False
        }
        model = XGBClassifier(**params)
    
    elif best_model_name == 'LightGBM':
        params = {
            'n_estimators': trial.suggest_int('n_estimators', 100, 300),
            'max_depth': trial.suggest_int('max_depth', 3, 10),
            'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3),
            'num_leaves': trial.suggest_int('num_leaves', 20, 100),
            'subsample': trial.suggest_float('subsample', 0.6, 1.0),
            'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 1.0),
            'random_state': 42,
            'verbose': -1
        }
        model = LGBMClassifier(**params)
    
    elif best_model_name == 'Gradient Boosting':
        params = {
            'n_estimators': trial.suggest_int('n_estimators', 100, 300),
            'max_depth': trial.suggest_int('max_depth', 3, 10),
            'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3),
            'subsample': trial.suggest_float('subsample', 0.6, 1.0),
            'random_state': 42
        }
        model = GradientBoostingClassifier(**params)
    
    else:  # Logistic Regression
        params = {
            'C': trial.suggest_float('C', 0.001, 100, log=True),
            'penalty': trial.suggest_categorical('penalty', ['l1', 'l2']),
            'solver': trial.suggest_categorical('solver', ['liblinear', 'saga']),
            'random_state': 42,
            'max_iter': 1000
        }
        model = LogisticRegression(**params)
    
    # Entrenar y evaluar
    model.fit(X_train_processed, y_train)
    y_val_proba = model.predict_proba(X_val_processed)[:, 1]
    auc = roc_auc_score(y_val, y_val_proba)
    
    return auc

# Ejecutar optimización
print("\n🔍 Ejecutando optimización (50 trials)...")
optuna.logging.set_verbosity(optuna.logging.WARNING)
study = optuna.create_study(direction='maximize', study_name='credit_risk_optimization')
study.optimize(objective, n_trials=50, show_progress_bar=True)

print(f"\n✓ Optimización completada")
print(f"\n🏆 Mejor AUC encontrado: {study.best_value:.4f}")
print(f"\n🔧 Mejores hiperparámetros:")
for key, value in study.best_params.items():
    print(f"  {key}: {value}")

# Visualizar historial de optimización
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Historial de trials
ax1 = axes[0]
trial_numbers = [trial.number for trial in study.trials]
trial_values = [trial.value for trial in study.trials]
ax1.plot(trial_numbers, trial_values, marker='o', linestyle='-', alpha=0.6, color='#3498db')
ax1.axhline(y=study.best_value, color='#e74c3c', linestyle='--', linewidth=2, label=f'Best: {study.best_value:.4f}')
ax1.set_xlabel('Trial', fontsize=12)
ax1.set_ylabel('Validation AUC', fontsize=12)
ax1.set_title('Historial de Optimización', fontsize=14, fontweight='bold')
ax1.legend()
ax1.grid(alpha=0.3)

# Importancia de hiperparámetros
ax2 = axes[1]
importances = optuna.importance.get_param_importances(study)
params_names = list(importances.keys())
importances_values = list(importances.values())
ax2.barh(params_names, importances_values, color='#2ecc71', alpha=0.8)
ax2.set_xlabel('Importancia', fontsize=12)
ax2.set_title('Importancia de Hiperparámetros', fontsize=14, fontweight='bold')
ax2.grid(axis='x', alpha=0.3)

plt.tight_layout()
plt.show()

In [0]:
# MODELO FINAL OPTIMIZADO
print("=" * 60)
print("🎯 MODELO FINAL OPTIMIZADO")
print("=" * 60)

# Crear modelo con mejores parámetros
if best_model_name == 'Random Forest':
    final_model = RandomForestClassifier(**study.best_params)
elif best_model_name == 'XGBoost':
    final_model = XGBClassifier(**study.best_params)
elif best_model_name == 'LightGBM':
    final_model = LGBMClassifier(**study.best_params)
elif best_model_name == 'Gradient Boosting':
    final_model = GradientBoostingClassifier(**study.best_params)
else:
    final_model = LogisticRegression(**study.best_params)

# Entrenar en train + validation
print("\n🚀 Entrenando modelo final en train + validation...")
X_train_full = np.vstack([X_train_processed, X_val_processed])
y_train_full = np.concatenate([y_train, y_val])

final_model.fit(X_train_full, y_train_full)

# Evaluar en test set
print("\n🧪 Evaluando en test set (datos nunca vistos)...")
y_test_pred = final_model.predict(X_test_processed)
y_test_proba = final_model.predict_proba(X_test_processed)[:, 1]

# Métricas
test_auc = roc_auc_score(y_test, y_test_proba)

print(f"\n{'='*60}")
print(f"🎉 AUC EN TEST SET: {test_auc:.4f}")
print(f"{'='*60}")

# Reporte de clasificación
print("\n📊 Reporte de Clasificación:")
print(classification_report(y_test, y_test_pred, target_names=['Bajo Riesgo', 'Alto Riesgo']))

# Matriz de confusión
fig, axes = plt.subplots(1, 2, figsize=(15, 6))

# Confusion Matrix
ax1 = axes[0]
cm = confusion_matrix(y_test, y_test_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax1, 
            xticklabels=['Bajo Riesgo', 'Alto Riesgo'],
            yticklabels=['Bajo Riesgo', 'Alto Riesgo'])
ax1.set_xlabel('Predicción', fontsize=12)
ax1.set_ylabel('Valor Real', fontsize=12)
ax1.set_title('Matriz de Confusión', fontsize=14, fontweight='bold')

# ROC Curve
ax2 = axes[1]
fpr, tpr, thresholds = roc_curve(y_test, y_test_proba)
ax2.plot(fpr, tpr, linewidth=3, label=f'ROC Curve (AUC = {test_auc:.4f})', color='#3498db')
ax2.plot([0, 1], [0, 1], 'k--', linewidth=2, label='Random Classifier')
ax2.set_xlabel('False Positive Rate', fontsize=12)
ax2.set_ylabel('True Positive Rate', fontsize=12)
ax2.set_title('Curva ROC', fontsize=14, fontweight='bold')
ax2.legend(loc='lower right')
ax2.grid(alpha=0.3)

plt.tight_layout()
plt.show()

# Registrar modelo final en MLflow
with mlflow.start_run(run_name=f"{best_model_name}_FINAL_OPTIMIZED"):
    mlflow.log_params(study.best_params)
    mlflow.log_metric("test_auc", test_auc)
    mlflow.sklearn.log_model(final_model, "final_model")
    print("\n✓ Modelo final registrado en MLflow")

In [0]:
# GENERACIÓN DE CREDIT SCORE
print("=" * 60)
print("💳 GENERACIÓN DE CREDIT SCORE POR CLIENTE")
print("=" * 60)

# Predecir probabilidades para todos los clientes
print("\n🔮 Calculando scores de riesgo para todos los clientes...")

# Procesar todo el dataset
X_all_processed = preprocessor.transform(X)
risk_probabilities = final_model.predict_proba(X_all_processed)[:, 1]

# Convertir probabilidad a score (0-1000)
# Score alto = bajo riesgo, Score bajo = alto riesgo
credit_scores = (1 - risk_probabilities) * 1000

# Clasificar en categorías de riesgo
def classify_risk(score):
    if score >= 800:
        return 'Excelente'
    elif score >= 700:
        return 'Muy Bueno'
    elif score >= 600:
        return 'Bueno'
    elif score >= 500:
        return 'Regular'
    else:
        return 'Alto Riesgo'

risk_categories = [classify_risk(score) for score in credit_scores]

# Crear DataFrame de resultados
credit_score_df = pd.DataFrame({
    'Cliente_ID': range(1, len(credit_scores) + 1),
    'Probabilidad_Default': risk_probabilities,
    'Credit_Score': credit_scores.astype(int),
    'Categoria_Riesgo': risk_categories,
    'Riesgo_Real': y
})

print(f"\n✓ Scores calculados para {len(credit_score_df)} clientes")

# Mostrar ejemplos
print("\n📊 Ejemplos de Credit Scores:")
print("\nClientes con mejor score (Menor Riesgo):")
display(credit_score_df.sort_values('Credit_Score', ascending=False).head(10))

print("\nClientes con peor score (Mayor Riesgo):")
display(credit_score_df.sort_values('Credit_Score', ascending=True).head(10))

# Distribución de scores
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# Histograma de scores
ax1 = axes[0, 0]
ax1.hist(credit_scores, bins=50, color='#3498db', alpha=0.7, edgecolor='black')
ax1.set_xlabel('Credit Score', fontsize=12)
ax1.set_ylabel('Frecuencia', fontsize=12)
ax1.set_title('Distribución de Credit Scores', fontsize=14, fontweight='bold')
ax1.axvline(credit_scores.mean(), color='red', linestyle='--', linewidth=2, label=f'Media: {credit_scores.mean():.0f}')
ax1.legend()
ax1.grid(alpha=0.3)

# Distribución por categoría
ax2 = axes[0, 1]
category_counts = credit_score_df['Categoria_Riesgo'].value_counts()
category_order = ['Excelente', 'Muy Bueno', 'Bueno', 'Regular', 'Alto Riesgo']
category_counts = category_counts.reindex(category_order, fill_value=0)
colors_cat = ['#27ae60', '#2ecc71', '#f39c12', '#e67e22', '#e74c3c']
ax2.bar(range(len(category_counts)), category_counts.values, color=colors_cat, alpha=0.8)
ax2.set_xticks(range(len(category_counts)))
ax2.set_xticklabels(category_counts.index, rotation=45, ha='right')
ax2.set_ylabel('Número de Clientes', fontsize=12)
ax2.set_title('Distribución por Categoría de Riesgo', fontsize=14, fontweight='bold')
ax2.grid(axis='y', alpha=0.3)

# Boxplot de scores por riesgo real
ax3 = axes[1, 0]
credit_score_df.boxplot(column='Credit_Score', by='Riesgo_Real', ax=ax3)
ax3.set_xlabel('Riesgo Real (0=Bajo, 1=Alto)', fontsize=12)
ax3.set_ylabel('Credit Score', fontsize=12)
ax3.set_title('Credit Score vs Riesgo Real', fontsize=14, fontweight='bold')
plt.sca(ax3)
plt.xticks([1, 2], ['Bajo Riesgo', 'Alto Riesgo'])

# Calibración: Score vs Probabilidad
ax4 = axes[1, 1]
ax4.scatter(credit_scores, risk_probabilities, alpha=0.3, color='#3498db', s=10)
ax4.set_xlabel('Credit Score', fontsize=12)
ax4.set_ylabel('Probabilidad de Default', fontsize=12)
ax4.set_title('Calibración: Score vs Probabilidad', fontsize=14, fontweight='bold')
ax4.grid(alpha=0.3)

plt.tight_layout()
plt.show()

print("\n📊 Estadísticas de Credit Score:")
print(credit_score_df['Credit_Score'].describe())

print("\n📊 Distribución por Categoría:")
print(credit_score_df['Categoria_Riesgo'].value_counts())

In [0]:
# GUARDAR RESULTADOS EN GOLD LAYER
print("=" * 60)
print("💾 GUARDANDO RESULTADOS EN GOLD LAYER")
print("=" * 60)

# Convertir a Spark DataFrame
credit_score_spark = spark.createDataFrame(credit_score_df)

# Crear schema si no existe
spark.sql("CREATE SCHEMA IF NOT EXISTS credit_risk")

# Guardar como tabla Delta (serverless no permite /FileStore/)
credit_score_spark.write.format('delta').mode('overwrite').saveAsTable('credit_risk.client_scores')

print(f"\n✓ Tabla Delta creada: credit_risk.client_scores")
print(f"✓ Total de registros: {credit_score_spark.count()}")
print("\n✅ Puedes consultar los scores con:")
print("   SELECT * FROM credit_risk.client_scores WHERE Categoria_Riesgo = 'Alto Riesgo'")

# Resumen final
print("\n" + "="*60)
print("🎉 PROYECTO COMPLETADO EXITOSAMENTE")
print("="*60)
print(f"\n📁 Arquitectura Medallion implementada:")
print(f"   ✅ Bronze: Datos crudos")
print(f"   ✅ Silver: Datos limpios y validados")
print(f"   ✅ Gold: Features y scores de ML")
print(f"\n🤖 Modelos entrenados: {len(models)}")
print(f"\n🎯 Mejor modelo: {best_model_name}")
print(f"   - Validation AUC: {best_val_auc:.4f}")
print(f"   - Test AUC: {test_auc:.4f}")
print(f"\n📊 Selección de variables: 3 métodos combinados")
print(f"   - ANOVA F-test")
print(f"   - Mutual Information")
print(f"   - Random Forest Importance")
print(f"\n⚙️ Optimización: Optuna (50 trials)")
print(f"\n💳 Credit scores generados: {len(credit_score_df)} clientes")
print(f"\n💾 Resultados guardados en Gold layer")